# Phase 1 — Feasibility Checks

**Purpose:** Determine whether the dataset supports the intended hierarchical model.
All four checks must pass or have confirmed fallbacks before proceeding.

In [ ]:
import pandas as pd

df1 = pd.read_excel("../data/online_retail_II.xlsx", sheet_name="Year 2009-2010")
df2 = pd.read_excel("../data/online_retail_II.xlsx", sheet_name="Year 2010-2011")
df = pd.concat([df1, df2], ignore_index=True)

print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
print(df.head())

## CHECK 1 — Country segment distribution

**This is the most important feasibility check.**

- **Go (ideal):** UK dominates but at least 8–10 countries have 50–500 unique customers.
- **Go (acceptable):** Fewer than 8 countries in middle tier — restrict to top 15.
- **No-go:** Fewer than 5 countries with >30 customers outside UK. Fallback: switch to product division grouping.

In [ ]:
seg = (
    df.groupby("Country")["Customer ID"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"Customer ID": "n_customers"})
)
print(seg.to_string())
print(f"\nTotal countries: {len(seg)}")
print(f"Countries with >50 customers: {(seg.n_customers > 50).sum()}")
print(f"Countries with 10-500 customers: {seg.n_customers.between(10,500).sum()}")

## CHECK 2 — Customer ID missingness

- **Go:** Overall missingness <35%. Drop rows with missing Customer ID.
- **Marginal (35–50%):** Proceed but flag in README. Check if missingness correlates with country/time.
- **No-go (>50%):** Switch to invoice-level repeat purchase rate.

**Research note:** Missing Customer ID likely reflects guest checkouts (MNAR). Do not impute.

In [ ]:
overall_missing = df["Customer ID"].isna().mean()
print(f"Overall Customer ID missing: {overall_missing:.1%}")

missing_by_country = (
    df.groupby("Country")["Customer ID"]
    .apply(lambda x: x.isna().mean())
    .sort_values(ascending=False)
)
print(missing_by_country)

## CHECK 3 — Churn base rate and class balance

- **Acceptable range:** 20–70% churn rate.
- If outside this range, adjust the churn window (try 60 or 120 days) and re-check.

In [ ]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
max_date = df["InvoiceDate"].max()
cutoff = max_date - pd.Timedelta(days=90)

customer_last = (
    df.dropna(subset=["Customer ID"])
    .groupby("Customer ID")["InvoiceDate"]
    .max()
    .reset_index()
    .rename(columns={"InvoiceDate": "last_purchase"})
)
customer_last["churned"] = (customer_last["last_purchase"] < cutoff).astype(int)

print(f"Churn base rate: {customer_last.churned.mean():.1%}")
print(f"Total identified customers: {len(customer_last)}")

## CHECK 4 — PyMC sampling speed

- **<120s:** Full model tractable with NUTS.
- **120–300s:** Tractable but slow. Use top 10 countries only.
- **>300s:** Switch to ADVI (variational inference).

In [ ]:
import pymc as pm
import numpy as np
import time

sample_df = customer_last.sample(5000, random_state=42)
outcome = sample_df["churned"].values
idx = (sample_df.index % 2).values

with pm.Model() as timing_model:
    mu = pm.Normal("mu", 0, 1)
    tau = pm.HalfNormal("tau", 1)
    beta = pm.Normal("beta", mu, tau, shape=2)
    p = pm.math.sigmoid(beta[idx])
    y = pm.Bernoulli("y", p=p, observed=outcome)
    
    t0 = time.time()
    trace = pm.sample(200, tune=200, cores=1, progressbar=True, random_seed=42)
    t1 = time.time()

print(f"Elapsed: {t1-t0:.0f}s for 200 samples + 200 tune, 5k obs, 2 segments")

## Phase 1 Exit Criteria Summary

Fill in after running all checks:

1. **Grouping variable:** Country / Product division
2. **Missingness rate:** __% — Handling: drop / flag / switch to invoice-level
3. **Churn window:** __ days — Base rate: __%
4. **Sampling strategy:** NUTS / ADVI